In [6]:
# FlagEmbedding을 이용해 임베딩 모델을 파인튜닝할 수 있습니다.
!pip install -U FlagEmbedding peft faiss-gpu LM_Cocktail

# 임베딩 모델의 파인튜닝 목적

허깅페이스(Hugging Face)에 공개되어 있는 대부분의 임베딩 모델은 일반화 성능이 뛰어납니다. 이는 다양한 도메인과 언어에 걸쳐 전반적으로 좋은 성능을 보인다는 의미입니다. 그러나 특정 비즈니스 분야나 사내 챗봇과 같이 전문화된 환경에서는 일반화된 모델보다 특화된 임베딩 모델이 더 효과적일 수 있습니다.

파인튜닝을 통해 임베딩 모델을 특정 도메인에 맞게 조정하면 다음과 같은 이점을 얻을 수 있습니다:

- 도메인 특화 언어 이해: 특정 분야의 용어와 표현을 더 잘 이해합니다.
- 유사도 측정 향상: 유사한 의미를 가진 문장을 더 정확하게 매칭합니다.
- 성능 개선: 검색, 추천 시스템 등에서 더 나은 결과를 제공합니다.

# 데이터 준비

임베딩 모델은 텍스트를 벡터화하여 수치 공간에서 표현합니다. 이때, 의미가 비슷한 문장들은 벡터 공간에서 가까이 위치하고, 의미가 다른 문장들은 멀리 위치하도록 학습합니다.

하지만 단어는 비슷하지만 의미가 다른 문장이 존재할 수 있습니다.


예를 들어:

1. "그녀는 은행에서 일한다."

여기서 **"은행"**은 금융기관을 의미합니다. 즉, 그녀는 돈과 관련된 업무를 하는 곳에서 일하고 있다는 뜻입니다.

2. "그는 강둑에서 낚시를 한다."

이 문장의 **"강둑"**을 영어로 하면 **"river bank"**입니다. 여기서 **"bank"**는 강의 둑이나 가장자리를 의미합니다.

Positive Sample: 원본 문장과 의미가 유사한 문장 쌍.
Negative Sample: 원본 문장과 의미가 다른 문장 쌍.

> 이를 방지하기 위해 Negative Sampling을 사용합니다:

Negative Sampling을 통해 모델이 의미적 차이를 더 잘 학습하도록 도와줍니다.

이렇게 동일하거나 비슷한 단어를 사용하지만 의미가 다른 문장들이 존재할 수 있습니다.
임베딩 모델이 이러한 문장들을 의미적으로 구분하지 못하고 단순히 단어의 유사성만으로 비슷하다고 판단할 수 있습니다.

> 그래서 Negative Sampling이 필요합니다:

Negative Sampling은 모델이 의미가 다른 문장들은 벡터 공간에서 멀리 떨어지도록 학습시키는 방법입니다.

위의 예시에서 두 문장은 단어는 같지만 의미는 다르므로, Negative Sample로 사용하여 모델이 이 둘을 유사하지 않다고 학습하도록 합니다.

이를 통해 모델이 단어의 표면적인 유사성에 의존하지 않고, 문장의 실제 의미를 파악할 수 있게 됩니다.



In [7]:
import json

# 번역된 데이터
translated_data = [
    {"query": "다섯 명의 여성이 해변을 따라 플립플롭을 신고 걸어간다.", "pos": ["플립플롭을 신은 몇몇 여성들이 해변을 따라 걸어가고 있다"], "neg": ["4명의 여성이 해변에 앉아 있다.", "1996년에 개혁이 있었다.", "그녀는 자신의 기록을 정정하기 위해 법정에 가지 않을 것이다.", "그 남자는 하와이에 대해 이야기하고 있다.", "한 여성이 밖에 서 있다.", "전투는 끝났다.", "한 무리의 사람들이 배구를 하고 있다."]},
    {"query": "한 여성이 높은 절벽 위에서 한 발로 서서 강을 내려다보고 있다.", "pos": ["한 여성이 절벽 위에 서 있다."], "neg": ["한 여성이 의자에 앉아 있다.", "조지 부시는 공화당원들에게 최고 고문들의 조언에 반하여 이 어리석은 생각을 고려조차 하지 않겠다고 말했다.", "그 가족은 무너지고 있었다.", "아무도 회의에 나타나지 않았다", "한 소년이 밖에서 모래를 가지고 놀고 있다.", "전보를 받자마자 끝났다.", "한 아이가 자기 방에서 책을 읽고 있다."]},
    {"query": "두 여성이 악기를 연주하고 있다; 한 명은 클라리넷, 다른 한 명은 바이올린을 연주한다.", "pos": ["몇 사람이 곡을 연주하고 있다."], "neg": ["두 여성이 기타와 드럼을 연주하고 있다.", "한 남자가 산을 스키를 타고 내려가고 있다.", "살인자가 생각했던 때에 치명적인 용량이 투여되지 않았다.", "자전거를 타고 있는 사람", "그 소녀는 아치길에 기대어 서 있다.", "한 무리의 여성들이 소파 오페라를 보고 있다.", "사람들은 나이가 들어도 절대 잊지 않는다."]},
    {"query": "파란색 탱크톱을 입은 소녀가 앉아서 세 마리의 개를 지켜보고 있다.", "pos": ["한 소녀가 파란색을 입고 있다."], "neg": ["한 소녀가 세 마리의 고양이와 함께 있다.", "사람들이 장례 행렬을 지켜보고 있다.", "그 아이는 검은색을 입고 있다.", "공립학교에서 우리에게 재정은 문제이다.", "수영장에 있는 아이들.", "폭행당하는 것은 진정시키는 일이다.", "나는 18살에 심각한 문제에 직면했다."]},
    {"query": "노란 개가 숲길을 따라 달리고 있다.", "pos": ["개가 달리고 있다"], "neg": ["고양이가 달리고 있다", "스틸은 그녀의 원래 이야기를 지키지 않았다.", "이 규칙은 사람들이 자녀 양육비를 내는 것을 막는다.", "조끼를 입은 남자가 차 안에 앉아 있다.", "검은 옷을 입고 흰색 반다나와 선글라스를 낀 사람이 버스 정류장에서 기다리고 있다.", "글로브나 메일 중 어느 쪽도 캐나다의 현재 도로 체계 상태에 대해 언급하지 않았다.", "스프링 크릭 시설은 오래되고 구식이다."]},
    {"query": "각 단계에서의 필수 활동과 그 활동들과 관련된 중요한 요소들을 설명한다.", "pos": ["필수 활동에 대한 중요 요소들이 설명되어 있다."], "neg": ["중요한 활동들을 설명하지만 그 활동들과 관련된 중요한 요소들에 대한 규정은 없다.", "사람들이 항의하기 위해 모여 있다.", "주 정부는 당신이 그렇게 하기를 선호할 것이다.", "한 소녀가 한 소년 옆에 앉아 있다.", "두 남성이 공연하고 있다.", "아무도 뛰고 있지 않다", "콘라드는 머리를 맞도록 음모를 꾸미고 있었다."]},
    {"query": "한 남자가 레스토랑에서 연설을 하고 있다.", "pos": ["한 사람이 연설을 하고 있다."], "neg": ["그 남자는 테이블에 앉아 음식을 먹고 있다.", "이것은 확실히 승인이 아니다.", "그들은 은퇴 때문에 집을 팔았지, 대출 때문이 아니다.", "미주리 주의 인장은 완벽하다.", "누군가가 손을 들고 있다.", "한 운동선수가 1500미터 수영 경기에 참가하고 있다.", "두 남자가 마술 쇼를 보고 있다."]},
    {"query": "인디언들이 코트를 입고 음식과 음료를 가지고 모임을 갖고 있다.", "pos": ["인디언 그룹이 음식과 음료를 가지고 모임을 갖고 있다"], "neg": ["인디언 그룹이 장례식을 하고 있다", "이것은 팔마의 큰 투우장에서 겨울 오후에만 공연된다.", "올바른 정보는 법률 서비스 관행과 사법 체계를 강화할 수 있다.", "한편, 본토는 인구가 없었다.", "두 아이가 자고 있다.", "어부가 원숭이를 잡으려고 하고 있다", "사람들이 기차 안에 있다"]},
    {"query": "보라색 머리를 한 여성이 밖에서 자전거를 타고 있다.", "pos": ["한 여성이 자전거를 타고 있다."], "neg": ["한 여성이 공원에서 조깅을 하고 있다.", "그 거리는 하얀색으로 칠해진 집들로 가득했다.", "한 그룹이 안에서 영화를 보고 있다.", "소풍에서 남자들이 스테이크를 자르고 있다", "여러 명의 요리사들이 앉아서 음식에 대해 이야기하고 있다.", "위원회는 중요한 대안들이 고려되지 않았다고 지적한다.", "우리는 장작이 다 떨어져서 불을 위해 소나무 바늘을 사용해야 했다."]},
    {"query": "한 남자가 도시 거리에서 인력거로 두 여성을 끌고 있다.", "pos": ["한 남자가 도시에 있다."], "neg": ["한 남자가 비행기 조종사이다.", "그것은 지루하고 평범하다.", "아침 햇살이 밝게 비치고 따뜻했다.", "두 사람이 부두에서 뛰어내렸다.", "사람들이 우주선 발사를 보고 있다.", "테레사 수녀는 쉬운 선택이다.", "원하는 속도로 갈 수 있는 것은 가치가 있다."]}
]

# JSONL 파일로 저장
with open('toy_finetune_data.jsonl', 'w', encoding='utf-8') as f:
    for item in translated_data:
        json.dump(item, f, ensure_ascii=False)
        f.write('\n')

print("번역된 데이터가 'toy_finetune_data.jsonl' 파일로 저장되었습니다.")

번역된 데이터가 'toy_finetune_data.jsonl' 파일로 저장되었습니다.


translated_data: 파인튜닝에 사용할 데이터 리스트입니다.
각 아이템은 딕셔너리로 구성되어 있습니다:
"query": 기준이 되는 문장입니다.
"pos":
"neg":

- query: 주요 쿼리 또는 질문입니다.  
- pos: 해당 쿼리에 대해 긍정적인(관련된) 텍스트들의 리스트입니다 -> "query" 와 의미가 유사한 문장들의 리스트입니다.
- neg: 해당 쿼리에 대해 부정적인(관련되지 않은) 텍스트들의 리스트입니다. 수백개 정도의 문장을 준비하는 것이 좋습니다 -> "query"와 의미가 다른 문장들의 리스트입니다.

> 일반적인 임베딩 파인튜닝 과정은 다음과 같습니다.
  
1. Positive (Pos) 설정

: 검색어(query)와 매칭될 수 있는 문서들을 Pos로 설정합니다. 이는 모델이 해당 문서들의 유사도를 높게 학습하도록 하기 위함입니다.


2.	Negative (Neg) 설정

: 반대로, 실제로 검색했을 때 연관성이 없지만 유사도가 높게 나오는 문장들을 Neg로 설정합니다. 이를 통해 모델이 이들 문서에 대한 유사도를 낮게 학습하도록 유도합니다.


3.	파인튜닝 진행

: 설정한 Pos와 Neg 데이터를 바탕으로 모델을 파인튜닝합니다. 이 과정에서 Pos에 포함된 문서들은 모델이 높은 유사도로 학습하도록, Neg는 낮은 유사도로 학습되도록 조정됩니다.

실제로 Query-Positive 관계는 신경 써서 잘 만들어야 합니다. 대신 Positive-Negative는 대략적으로라도 많이 만들어서 넣어주는 것이 중요합니다.

FlagEmbedding 패키지가 Negative 데이터들을 의미있는 데이터로 바꿔줄 수 있습니다.

임베딩 모델의 파인튜닝은 특정 비즈니스 요구사항에 맞는 맞춤형 모델을 만드는 데 필수적입니다.

올바른 데이터 준비와 Negative Sampling 기법을 통해 모델이 의미적 유사성과 차이를 정확하게 학습하도록 할 수 있습니다.

## Hard Negatives (선택적)


* Hard Negatives란?

Hard Negatives는 모델이 학습하기 어려운 **'어려운 부정 샘플'**을 의미합니다.

일반적인 Negative Sample보다 원본 문장과 유사하지만 실제로는 다른 의미를 가진 문장들입니다.

이러한 샘플을 사용하면 모델이 미묘한 의미 차이를 구별하도록 훈련되어 성능이 향상됩니다.

**선택적인 옵션입니다. 직접 사람이 만든 것 보다 임베딩 모델이 한 것을 사용하고 싶을 때 사용할 수 있습니다.**


* Hard Negatives 생성 방법
우리는 이미 준비한 데이터에 대해 기존 임베딩 모델을 사용하여 Hard Negatives를 생성할 수 있습니다.

목적

: 기존 모델이 유사하다고 판단하지만 실제로는 부정적인 샘플을 찾아내어 모델이 더 정교하게 학습되도록 합니다.

방법

: 각 쿼리에 대해 기존 임베딩 모델로 유사도가 높은 문장들 중에서 부정 샘플을 선택합니다.


```bash
!python -m FlagEmbedding.baai_general_embedding.finetune.hn_mine \
--model_name_or_path BAAI/bge-m3 \ # query 하나랑 전체 json 데이터의 negative 데이터랑 유사도 계산. 그 다음 임베딩 유사도가 높은 순으로 Negative를 덮어쓰기를 한다.
--input_file toy_finetune_data.jsonl \
--output_file toy_finetune_data_minedHN.jsonl \
--range_for_sampling 2-200 \
--negative_number 15 \ # negative 데이터의 개수. 원래 첫 번째는 7개인데, 유사도 순으로 정렬해서 높은 순으로 15개 넣음.
--use_gpu_for_searching
```

In [11]:
!python -m FlagEmbedding.baai_general_embedding.finetune.hn_mine \
--model_name_or_path BAAI/bge-m3 \
--input_file toy_finetune_data.jsonl \
--output_file toy_finetune_data_minedHN.jsonl \
--range_for_sampling 2-200 \
--negative_number 15 \
--use_gpu_for_searching

2024-10-16 04:26:31.668141: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-16 04:26:31.703291: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-16 04:26:31.714557: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-16 04:26:31.742223: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-16 04:26:33.481148: W tensorflow/compiler/tf2

* `input_file`: 최적화 하기를 원하는 여러분들이 파인튜닝을 위해 준비한 JSON 데이터입니다. 위 코드를 실행하면 각 `query`에 대해서 실제로 `bge-m3`로 유사도가 높은 상위 k개의 문서를 검색하고, 이 상위 k개 문서에서 무작위로 네거티브 샘플링을 수행합니다 (`pos` 문서는 제외).

* `output_file`: 네거티브 샘플링을 수행하여 여러분들의 데이터를 학습하기에 최적화 시킨 후의 JSON 데이터를 저장할 경로입니다.

* `negative_number`: 샘플링할 네거티브의 수입니다.

* `range_for_sampling`: 네거티브를 샘플링할 범위입니다. 예를 들어, `2-100`은 상위 2위부터 200위 문서 중에서 `negative_number`만큼의 네거티브를 샘플링한다는 의미입니다. **네거티브의 난이도를 낮추기 위해 더 큰 값을 설정할 수 있습니다 (예: 상위 60-300위 문서에서 네거티브를 샘플링하려면 `60-300`으로 설정)**

* `use_gpu_for_searching`: 네거티브 검색에 faiss-gpu를 사용할지 여부입니다.

코드 실행 후에는 `toy_finetune_data_minedHN.jsonl` 라는 새로운 파일이 생기게 되며 해당 파일은 우리가 기존에 만들어두었던 `toy_finetune_data.jsonl` 파일과는 내용이 다릅니다. 예를 들어 변경 전의 첫번째 데이터와 변경 후의 첫번째 데이터를 비교해봅시다.

In [ ]:
print("=========== 변경 전 ===========")
with open('toy_finetune_data.jsonl', 'r', encoding='utf-8') as f:
  first_data = json.loads(f.readline())

print(first_data, "\nnegative 샘플 개수 : {}".format(len(first_data['neg'])))

print()
print("=========== 변경 후 ===========")

with open('toy_finetune_data_minedHN.jsonl', 'r', encoding='utf-8') as f:
  first_data = json.loads(f.readline())

print(first_data, "\nnegative 샘플 개수 : {}".format(len(first_data['neg'])))

=========== 변경 전 ===========
{'query': '다섯 명의 여성이 해변을 따라 플립플롭을 신고 걸어간다.', 'pos': ['플립플롭을 신은 몇몇 여성들이 해변을 따라 걸어가고 있다'], 'neg': ['4명의 여성이 해변에 앉아 있다.', '1996년에 개혁이 있었다.', '그녀는 자신의 기록을 정정하기 위해 법정에 가지 않을 것이다.', '그 남자는 하와이에 대해 이야기하고 있다.', '한 여성이 밖에 서 있다.', '전투는 끝났다.', '한 무리의 사람들이 배구를 하고 있다.']} 
negative 샘플 개수 : 7

=========== 변경 후 ===========
{'query': '다섯 명의 여성이 해변을 따라 플립플롭을 신고 걸어간다.', 'pos': ['플립플롭을 신은 몇몇 여성들이 해변을 따라 걸어가고 있다'], 'neg': ['그것은 지루하고 평범하다.', '한 남자가 도시에 있다.', '한 여성이 의자에 앉아 있다.', '사람들이 장례 행렬을 지켜보고 있다.', '이것은 확실히 승인이 아니다.', '중요한 활동들을 설명하지만 그 활동들과 관련된 중요한 요소들에 대한 규정은 없다.', '위원회는 중요한 대안들이 고려되지 않았다고 지적한다.', '한 소녀가 한 소년 옆에 앉아 있다.', '사람들이 기차 안에 있다', '사람들이 항의하기 위해 모여 있다.', '두 여성이 기타와 드럼을 연주하고 있다.', '한 여성이 밖에 서 있다.', '인디언 그룹이 장례식을 하고 있다', '한 무리의 사람들이 배구를 하고 있다.', '그 소녀는 아치길에 기대어 서 있다.']} 
negative 샘플 개수 : 15


# 훈련

In [ ]:
!torchrun --nproc_per_node 1 \
-m FlagEmbedding.baai_general_embedding.finetune.run \
--output_dir ./checkpoint \
--model_name_or_path BAAI/bge-m3 \
--train_data ./toy_finetune_data_minedHN.jsonl \
--learning_rate 1e-5 \
--fp16 \
--num_train_epochs 5 \
--per_device_train_batch_size 1 \
--dataloader_drop_last True \
--normlized True \
--temperature 0.02 \
--query_max_len 64 \
--passage_max_len 256 \
--train_group_size 2 \
--negatives_cross_device \
--logging_steps 10 \
--save_steps 1000 \
--query_instruction_for_retrieval ""

2024-10-16 02:41:39.030288: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-16 02:41:39.047940: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-16 02:41:39.069076: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-16 02:41:39.075405: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-16 02:41:39.090435: I tensorflow/core/platform/cpu_feature_guar

* `per_device_train_batch_size`: 학습 시 배치 크기입니다. 대부분의 경우, 더 큰 배치 크기가 더 강력한 성능을 가져옵니다. `--fp16`, `--deepspeed ./df_config.json` (df_config.json은 ds_config.json을 참조할 수 있음), `--gradient_checkpointing` 등을 활성화하여 확장할 수 있습니다.

* `train_group_size`: 학습 시 쿼리당 긍정 및 부정 예제의 수입니다. 항상 하나의 긍정 예제가 있으므로, 이 인자는 부정 예제의 수를 제어합니다 (부정 예제 수 = train_group_size - 1). 부정 예제의 수가 데이터의 `"neg":List[str]`에 있는 부정 예제 수보다 크지 않아야 함에 주의하세요. 이 그룹의 부정 예제 외에도, 배치 내 부정 예제도 파인튜닝에 사용됩니다.

* `negatives_cross_device`: 모든 GPU에서 부정 예제를 공유합니다. 이 인자는 부정 예제의 수를 확장합니다.

* `learning_rate`: 모델에 적합한 값을 선택하세요. 대규모/기본/소규모 모델에 대해 각각 1e-5/2e-5/3e-5를 추천합니다.

* `temperature`: 유사도 점수의 분포에 영향을 미칩니다. **권장 값: 0.01-0.1.**

* `query_max_len`: 쿼리의 최대 길이입니다. 데이터의 평균 쿼리 길이에 따라 설정해 주세요.

* `passage_max_len`: 문장의 최대 길이입니다. 데이터의 평균 문장 길이에 따라 설정해 주세요.

* `query_instruction_for_retrieval`: 쿼리에 대한 지시사항으로, 각 쿼리에 추가됩니다. 아무것도 추가하지 않으려면 `""`로 설정할 수 있습니다.

* `use_inbatch_neg`: 같은 배치 내의 문장들을 부정 예제로 사용합니다. 기본값은 True입니다.

* `save_steps`: 체크포인트를 저장할 학습 단계 간격을 설정합니다.

# 파인튜닝 된 모델 불러오기 & 사용

In [ ]:
from FlagEmbedding import FlagModel

sentences_1 = ["다섯 명의 여성이 해변을 따라 플립플롭을 신고 걸어간다."] # 원래 학습데이터에 있던 쿼리

# 첫 번째 문장은 pos, 두 번째 문장은 neg
sentences_2 = ["플립플롭을 신은 몇몇 여성들이 해변을 따라 걸어가고 있다", "꽁꽁 얼어붙은 한강 위로 고양이가 걸어가고 있다"]

# 기존 모델 불러오기(파인튜닝이 되지 않은 bge-m3)
base_model = FlagModel("BAAI/bge-m3", use_fp16=True)

# 파인튜닝된 모델 불러오기
fine_tuned_model = FlagModel("./checkpoint", use_fp16=True)

# 파인 튜닝이 되지 않은 기존 모델로 유사도 구하기
embeddings_1 = base_model.encode(sentences_1)
embeddings_2 = base_model.encode(sentences_2)

base_model_similarity = embeddings_1 @ embeddings_2.T

# 파인 튜닝이 된 모델로 유사도 구하기
embeddings_1_fine_tuned = fine_tuned_model.encode(sentences_1)
embeddings_2_fine_tuned = fine_tuned_model.encode(sentences_2)

fine_tuned_model_similarity = embeddings_1_fine_tuned @ embeddings_2_fine_tuned.T

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
print("기존 모델의 유사도:", base_model_similarity)
print("파인 튜닝 된 모델의 유사도:", fine_tuned_model_similarity)

기존 모델의 유사도: [[0.9336 0.4075]]
파인 튜닝 된 모델의 유사도: [[0.919 0.344]]


# Model merging via LM-Cocktail (선택적)

기본 BGE 모델을 파인튜닝하면 목표 작업에서의 성능을 향상시킬 수 있지만, 대상 도메인을 넘어선 모델의 일반적 능력이 심각하게 저하될 수 있습니다(예: c-mteb 작업에서의 성능 저하). 파인튜닝된 모델과 기본 모델을 병합함으로써, LM-Cocktail은 다른 관련 없는 작업에서의 성능을 유지하면서도 다운스트림 작업에서의 성능을 크게 향상시킬 수 있습니다.

정리하자면 너무 한쪽 비즈니스에 특화시킨 모델은 다른 비즈니스에 대한 성능이 너무 안좋아질 수 있기 때문에 기본 모델 + 특화 시킨 모델을 합치는 기법입니다.

In [ ]:
from LM_Cocktail import mix_models, mix_models_with_data

model = mix_models(
    model_names_or_paths=["BAAI/bge-m3", "./checkpoint"],
    model_type="encoder",
    weights=[0.5, 0.5],
    output_path="./mix_model_1"
)

loading BAAI/bge-m3 -----------------
loading ./checkpoint -----------------
***weight for each model***: 
BAAI/bge-m3 0.5
./checkpoint 0.5
Saving the new model to ./mix_model_1
Transform the model to the format of 'sentence_transformers' (pooling_method='cls', normalized=True)


In [ ]:
from FlagEmbedding import FlagModel

sentences_1 = ["다섯 명의 여성이 해변을 따라 플립플롭을 신고 걸어간다."]
sentences_2 = ["플립플롭을 신은 몇몇 여성들이 해변을 따라 걸어가고 있다", "꽁꽁 얼어붙은 한강 위로 고양이가 걸어가고 있다"]

# Setting use_fp16 to True speeds up computation with a slight performance degradation
model = FlagModel('./mix_model_1', use_fp16=True)

embeddings_1 = model.encode(sentences_1)
embeddings_2 = model.encode(sentences_2)
similarity = embeddings_1 @ embeddings_2.T

# ⭐️ 믹스 모델은 파인튜닝 하지 않았을 때와 파인 튜닝 했을 때의 중간 정도의 성능이 나온다.
print('믹스 모델:', similarity)

믹스 모델: [[0.9263 0.3591]]


```
기존 모델의 유사도: [[0.9336 0.4075]]
파인 튜닝 된 모델의 유사도: [[0.919 0.344]]
```

새로운 작업이 있고, 파인튜닝에 사용할 수 있는 데이터나 리소스가 없는 경우, LM-Cocktail을 사용하여 기존 모델들(오픈소스 커뮤니티의 모델이나 다른 작업에 대해 파인튜닝한 여러분의 모델들)을 병합하여 작업 특화 모델을 만들 수 있습니다. 이 방법을 사용하면 소수의 예시 데이터만 구성하면 되고, 기본 모델을 파인튜닝할 필요가 없습니다. 예를 들어, 여러분의 작업에 대한 예시 데이터를 사용하여 Hugging Face의 모델들을 병합할 수 있습니다.

In [ ]:
from LM_Cocktail import mix_models, mix_models_with_data

example_data = [
    {"query": "텔루구 영화 산업에서 배우가 되려면 어떻게 해야 하나요?", "pos": ["텔루구 영화 산업에서 배우가 되려면 어떻게 해야 하나요?"], "neg": ["모세와 람세스의 이야기는 무엇인가요?", "카스트 제도가 인도의 경제 성장에 영향을 미치나요?"]},
    {"query": "왜 어떤 컴퓨터 프로그래머들은 놀라운 소프트웨어나 새로운 개념을 개발하는 반면, 어떤 이들은 기본적인 프로그래밍 작업에 머물러 있나요?", "pos": ["왜 어떤 컴퓨터 프로그래머들은 놀라운 소프트웨어나 새로운 개념을 개발하는 반면, 어떤 이들은 기본적인 프로그래밍 작업에 머물러 있나요?"], "neg": ["친구를 방문할 때, 그들을 때리거나 가구를 부수는 등의 매우 부적절한 행동을 하면 어떻게 될지 생각해 본 적이 있나요?", "칭찬과 flirting의 차이점은 무엇인가요?"]}
]

print(example_data)

[{'query': '텔루구 영화 산업에서 배우가 되려면 어떻게 해야 하나요?', 'pos': ['텔루구 영화 산업에서 배우가 되려면 어떻게 해야 하나요?'], 'neg': ['모세와 람세스의 이야기는 무엇인가요?', '카스트 제도가 인도의 경제 성장에 영향을 미치나요?']}, {'query': '왜 어떤 컴퓨터 프로그래머들은 놀라운 소프트웨어나 새로운 개념을 개발하는 반면, 어떤 이들은 기본적인 프로그래밍 작업에 머물러 있나요?', 'pos': ['왜 어떤 컴퓨터 프로그래머들은 놀라운 소프트웨어나 새로운 개념을 개발하는 반면, 어떤 이들은 기본적인 프로그래밍 작업에 머물러 있나요?'], 'neg': ['친구를 방문할 때, 그들을 때리거나 가구를 부수는 등의 매우 부적절한 행동을 하면 어떻게 될지 생각해 본 적이 있나요?', '칭찬과 flirting의 차이점은 무엇인가요?']}]


In [ ]:
model = mix_models_with_data(
    model_names_or_paths=["BAAI/bge-base-en-v1.5", "Shitao/bge-hotpotqa", "Shitao/bge-quora"],
    model_type='encoder',
    example_ata=example_data,
    temperature=5.0,
    max_input_length=512,
    neg_number=2)

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

loading Shitao/bge-quora -----------------


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

***weight for each model***: 
BAAI/bge-base-en-v1.5 0.3306178152561188
Shitao/bge-hotpotqa 0.33525559306144714
Shitao/bge-quora 0.3341265618801117
